# Mask2Former — Fine-tune trên dataset lá **Cà phê** (Coffee)

**Domain:** Coffee — 4 classes: LeafMiner, PowderyMildew, Rust, AlgalLeafSpot
**Kaggle Dataset:** magnusdtd2/rice-coffee-leaf-disease
**Reference:** https://debuggercafe.com/fine-tuning-mask2former/

## Nội dung
1. Setup & Imports
2. Cấu hình (seed, paths, hyperparameters)
3. Data — COCO loader + Dataset/DataLoader
4. Augmentation (Affine, Intensity, CutOut, CutMix, Mixup)
5. Model — Mask2FormerForUniversalSegmentation
6. Training loop
7. Evaluation — mAP@50, mAP@50:95, mIoU, Dice, inference time
8. Hyperparameter tuning — Run #1 vs Run #2
9. Visualization (≥ 10 ảnh test: pred vs GT)
10. Error analysis
11. Push checkpoint lên HuggingFace Hub
12. Bảng so sánh trước/sau tuning

## 1. Setup & Imports

In [ ]:
%pip install -q --upgrade transformers accelerate timm albumentations pycocotools torchmetrics huggingface_hub scikit-learn

In [ ]:
import os
import json
import random
import time
from collections import Counter
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw

import albumentations as A
from albumentations.pytorch import ToTensorV2
from pycocotools import mask as mask_utils

from transformers import (
    Mask2FormerForUniversalSegmentation,
    Mask2FormerImageProcessor,
)
from torchmetrics.detection import MeanAveragePrecision

import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm

print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## 2. Cấu hình

In [ ]:
SEED = 42
DOMAIN = 'coffee'
DATASET_SLUG = 'rice-coffee-leaf-disease'
DOMAIN_FOLDER = 'coffee_leaf_disease'


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

# Coffee: 4 classes (theo src/utils/config.py: COFFEE_CLASSES)
# Thứ tự này khớp docs/ml-data-report.md:
#   0 -> LeafMiner, 1 -> PowderyMildew, 2 -> Rust, 3 -> AlgalLeafSpot
CLASS_NAMES = ['LeafMiner', 'PowderyMildew', 'Rust', 'AlgalLeafSpot']
COFFEE_FOLDER_TO_LABEL = {str(i): name for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)
ID2LABEL = {i: name for i, name in enumerate(CLASS_NAMES)}
LABEL2ID = {name: i for i, name in enumerate(CLASS_NAMES)}


def find_domain_root(domain_folder: str = DOMAIN_FOLDER) -> Path:
    """Find coffee_leaf_disease root on Kaggle or local repo.

    Expected Kaggle structure:
      /kaggle/input/rice-coffee-leaf-disease/coffee_leaf_disease/annotations.coco.json

    The function also supports nested exports such as:
      /kaggle/input/rice-coffee-leaf-disease/datasets/final/coffee_leaf_disease/...
    """
    candidates = [
        Path('/kaggle/input') / DATASET_SLUG / domain_folder,
        Path('/kaggle/input') / DATASET_SLUG / 'datasets' / 'final' / domain_folder,
        Path('/kaggle/input') / DATASET_SLUG / 'final' / domain_folder,
        Path('/kaggle/input') / domain_folder,
        Path.cwd() / 'datasets' / 'final' / domain_folder,
    ]
    for candidate in candidates:
        if (candidate / 'annotations.coco.json').exists():
            return candidate

    input_root = Path('/kaggle/input')
    if input_root.exists():
        matches = sorted(input_root.rglob(f'{domain_folder}/annotations.coco.json'))
        if matches:
            return matches[0].parent

    raise FileNotFoundError(
        f"Could not find {domain_folder}/annotations.coco.json. "
        "On Kaggle, click Add Data and attach magnusdtd2/rice-coffee-leaf-disease."
    )


DATA_ROOT = find_domain_root()
ANNO_JSON = DATA_ROOT / 'annotations.coco.json'
IMG_ROOT = DATA_ROOT  # COCO file_name is relative to this root, e.g. "0/img.jpg"

OUTPUT_DIR = Path('/kaggle/working/mask2former_coffee') if Path('/kaggle/working').exists() else Path('outputs/mask2former_coffee')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Domain: {DOMAIN} | Classes: {CLASS_NAMES} | Device: {DEVICE}')
print(f'DATA_ROOT: {DATA_ROOT}')
print(f'ANNO_JSON exists: {ANNO_JSON.exists()}')
print('Top-level files/folders:', sorted(p.name for p in DATA_ROOT.iterdir())[:12])

In [ ]:
# Hyperparameters: 2 cấu hình (≥ 2 theo yêu cầu PLAN.md)
# Để smoke test nhanh trên Kaggle, đổi FAST_DEV_RUN = True rồi chạy toàn bộ notebook.
FAST_DEV_RUN = False

RUN_CONFIGS = {
    'run1_baseline': {
        'backbone': 'facebook/mask2former-swin-tiny-coco-instance',
        'image_size': 384,
        'batch_size': 4,
        'lr': 5e-5,
        'weight_decay': 1e-4,
        'epochs': 30,
        'aug_level': 'light',
    },
    'run2_tuned': {
        'backbone': 'facebook/mask2former-swin-small-coco-instance',
        'image_size': 512,
        'batch_size': 2,
        'lr': 1e-4,
        'weight_decay': 5e-5,
        'epochs': 30,
        'aug_level': 'strong',
    },
}

if FAST_DEV_RUN:
    RUN_CONFIGS = {
        'debug_smoke': {
            'backbone': 'facebook/mask2former-swin-tiny-coco-instance',
            'image_size': 256,
            'batch_size': 1,
            'lr': 5e-5,
            'weight_decay': 1e-4,
            'epochs': 1,
            'aug_level': 'light',
            'max_train_batches': 2,
            'max_val_batches': 1,
            'max_eval_samples': 3,
        },
    }

RUN_CONFIGS

## 3. Data — COCO loader + Stratified Split + Dataset

Kaggle dataset có **1 file `annotations.coco.json`** per domain (chưa split).
- Coffee: images ở subfolders `0/`, `1/`, `2/`, `3/`.
- Mapping theo project docs: `0=LeafMiner`, `1=PowderyMildew`, `2=Rust`, `3=AlgalLeafSpot`.
- `file_name` trong COCO JSON thường là `"0/image.jpg"` → đọc ảnh tại `IMG_ROOT / file_name`.
- COCO categories có thể dùng tên số hoặc tên đầy đủ — `PlantSegDataset` xử lý cả hai.
- Split stratified 70/15/15 (seed=42) thực hiện in-memory để tránh lỗi thiếu `train.coco.json`.

In [ ]:
def load_coco(json_path) -> dict:
    with open(json_path, 'r', encoding='utf-8') as f:
        return json.load(f)


def segmentation_to_mask(segmentation, height: int, width: int) -> np.ndarray:
    """Convert COCO polygon/RLE segmentation to a binary uint8 mask."""
    if isinstance(segmentation, list):
        mask = Image.new('L', (width, height), 0)
        draw = ImageDraw.Draw(mask)
        for poly in segmentation:
            if len(poly) >= 6:
                draw.polygon(poly, outline=1, fill=1)
        return np.array(mask, dtype=np.uint8)

    if isinstance(segmentation, dict):
        decoded = mask_utils.decode(segmentation)
        if decoded.ndim == 3:
            decoded = decoded.any(axis=2)
        return decoded.astype(np.uint8)

    return np.zeros((height, width), dtype=np.uint8)


def category_to_label_idx(category: dict, fallback_order: dict) -> int | None:
    """Map a COCO category to project label index.

    Supports:
    - category name is full disease name, e.g. "Rust"
    - category name is numeric folder id, e.g. "2"
    - category id itself is 0..3 or 1..4 as a fallback
    """
    name = str(category.get('name', '')).strip()
    if name in LABEL2ID:
        return LABEL2ID[name]
    if name in COFFEE_FOLDER_TO_LABEL:
        return LABEL2ID[COFFEE_FOLDER_TO_LABEL[name]]

    cat_id = category.get('id')
    if str(cat_id) in COFFEE_FOLDER_TO_LABEL:
        return LABEL2ID[COFFEE_FOLDER_TO_LABEL[str(cat_id)]]
    if isinstance(cat_id, int) and 1 <= cat_id <= NUM_CLASSES:
        return cat_id - 1

    return fallback_order.get(cat_id)


class PlantSegDataset(Dataset):
    def __init__(self, image_dir, coco_data, transform=None):
        """
        image_dir : root of images (IMG_ROOT); file_name in COCO is relative to this
        coco_data : pre-loaded COCO dict (from split_coco_by_ids) OR Path/str to JSON file
        """
        self.image_dir = Path(image_dir)
        self.coco = load_coco(coco_data) if isinstance(coco_data, (str, Path)) else coco_data
        self.transform = transform

        self.images = {img['id']: img for img in self.coco['images']}

        fallback_order = {
            cat['id']: idx
            for idx, cat in enumerate(sorted(self.coco['categories'], key=lambda c: c['id']))
            if idx < NUM_CLASSES
        }
        self.cat_id_to_label_idx: Dict[int, int] = {}
        for cat in self.coco['categories']:
            label_idx = category_to_label_idx(cat, fallback_order)
            if label_idx is not None:
                self.cat_id_to_label_idx[cat['id']] = label_idx

        if not self.cat_id_to_label_idx:
            raise ValueError(f'Could not map COCO categories to labels: {self.coco["categories"]}')

        self.img_to_anns: Dict[int, List[dict]] = {}
        for ann in self.coco['annotations']:
            if ann.get('category_id') in self.cat_id_to_label_idx:
                self.img_to_anns.setdefault(ann['image_id'], []).append(ann)

        self.image_ids = [
            iid for iid in self.images
            if iid in self.img_to_anns and len(self.img_to_anns[iid]) > 0
        ]

    def __len__(self):
        return len(self.image_ids)

    def resolve_image_path(self, file_name: str) -> Path:
        file_name = file_name.replace('\\', '/')
        candidates = [
            self.image_dir / file_name,
            self.image_dir / Path(file_name).name,
            self.image_dir.parent / file_name,
        ]
        for candidate in candidates:
            if candidate.exists():
                return candidate
        raise FileNotFoundError(
            f'Could not find image {file_name}. Tried: {[str(c) for c in candidates]}'
        )

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        info = self.images[img_id]
        image_path = self.resolve_image_path(info['file_name'])
        image = np.array(Image.open(image_path).convert('RGB'))
        h, w = image.shape[:2]

        masks, labels = [], []
        for ann in self.img_to_anns[img_id]:
            cat_id = ann.get('category_id')
            seg = ann.get('segmentation')
            if cat_id not in self.cat_id_to_label_idx:
                continue
            m = segmentation_to_mask(seg, h, w)
            if m.sum() > 0:
                masks.append(m)
                labels.append(self.cat_id_to_label_idx[cat_id])

        if len(masks) == 0:
            masks = [np.zeros((h, w), dtype=np.uint8)]
            labels = [0]

        if self.transform is not None:
            out = self.transform(image=image, masks=masks)
            image, masks = out['image'], out['masks']

        return image, masks, labels

In [ ]:
# ─── Stratified 70/15/15 split từ single annotations.coco.json ────────────────────────
from sklearn.model_selection import train_test_split as _tts


def split_coco_by_ids(coco: dict, image_ids: list) -> dict:
    keep = set(image_ids)
    return {
        'images': [img for img in coco['images'] if img['id'] in keep],
        'annotations': [ann for ann in coco['annotations'] if ann['image_id'] in keep],
        'categories': coco['categories'],
    }


def label_name_from_cat_id(cat_id, categories) -> str:
    cat = next((c for c in categories if c['id'] == cat_id), None)
    if cat is None:
        return f'unknown_{cat_id}'
    idx = category_to_label_idx(cat, {})
    return ID2LABEL[idx] if idx is not None else str(cat.get('name', cat_id))


full_coco = load_coco(ANNO_JSON)
print('COCO keys:', sorted(full_coco.keys()))
print('Categories:', full_coco['categories'])

img_id_to_cat: dict = {}
for ann in full_coco['annotations']:
    iid = ann['image_id']
    if iid not in img_id_to_cat:
        img_id_to_cat[iid] = ann['category_id']

all_ids = list(img_id_to_cat.keys())
all_cats = [img_id_to_cat[iid] for iid in all_ids]
assert all_ids, 'No annotated images found in annotations.coco.json'

train_ids, tmp_ids, train_cats, tmp_cats = _tts(
    all_ids, all_cats, test_size=0.30, stratify=all_cats, random_state=SEED
)
val_ids, test_ids, _, _ = _tts(
    tmp_ids, tmp_cats, test_size=0.50, stratify=tmp_cats, random_state=SEED
)

train_coco = split_coco_by_ids(full_coco, train_ids)
val_coco = split_coco_by_ids(full_coco, val_ids)
test_coco = split_coco_by_ids(full_coco, test_ids)

split_rows = []
for split_name, ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)]:
    counts = Counter(label_name_from_cat_id(img_id_to_cat[iid], full_coco['categories']) for iid in ids)
    for cls in CLASS_NAMES:
        split_rows.append({'split': split_name, 'class': cls, 'count': counts.get(cls, 0)})

split_df = pd.DataFrame(split_rows)
display(split_df.pivot(index='class', columns='split', values='count').fillna(0).astype(int))
print(f'Total images with annotations: {len(all_ids)}')
print(f'Split — train: {len(train_ids)} ({len(train_ids)/len(all_ids):.0%}) '
      f'| val: {len(val_ids)} ({len(val_ids)/len(all_ids):.0%}) '
      f'| test: {len(test_ids)} ({len(test_ids)/len(all_ids):.0%})')

# Quick image/mask sanity check before launching long training.
_sanity_ds = PlantSegDataset(IMG_ROOT, train_coco, transform=build_transform(256, 'light', False) if 'build_transform' in globals() else None)
print('Mapped category_id -> label:', {k: ID2LABEL[v] for k, v in _sanity_ds.cat_id_to_label_idx.items()})
print('Train dataset samples:', len(_sanity_ds))

## 4. Augmentation

Theo `docs/PLAN.md` dòng 96: Affine, Intensity Transformation, CutOut, CutMix, Mixup.

In [ ]:
def build_transform(image_size: int, level: str = 'light', training: bool = True):
    pad_kwargs = {
        'min_height': image_size,
        'min_width': image_size,
        'border_mode': 0,
        'p': 1.0,
    }
    # Albumentations v1 uses value/mask_value; v2 uses fill/fill_mask.
    try:
        A.PadIfNeeded(value=0, mask_value=0, **pad_kwargs)
        pad = A.PadIfNeeded(value=0, mask_value=0, **pad_kwargs)
    except TypeError:
        pad = A.PadIfNeeded(fill=0, fill_mask=0, **pad_kwargs)

    ops = [
        A.LongestMaxSize(max_size=image_size),
        pad,
    ]
    if training:
        ops += [A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.2)]
        if level == 'light':
            ops += [
                A.Affine(scale=(0.9, 1.1), translate_percent=(0, 0.05),
                         rotate=(-15, 15), p=0.5),
                A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
            ]
        elif level == 'strong':
            ops += [
                A.Affine(scale=(0.8, 1.2), translate_percent=(0, 0.1),
                         rotate=(-30, 30), shear=(-10, 10), p=0.7),
                A.RandomBrightnessContrast(0.3, 0.3, p=0.5),
                A.HueSaturationValue(10, 20, 10, p=0.4),
                A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),  # CutOut
            ]
    ops += [
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ]
    return A.Compose(ops)


def rand_bbox(h, w, lam):
    cut = np.sqrt(1.0 - lam)
    cw, ch = int(w * cut), int(h * cut)
    cx, cy = np.random.randint(w), np.random.randint(h)
    return max(cx - cw // 2, 0), max(cy - ch // 2, 0), min(cx + cw // 2, w), min(cy + ch // 2, h)


def apply_cutmix_pixels(pixel_values: torch.Tensor, lam: float = None):
    if lam is None:
        lam = float(np.random.beta(1.0, 1.0))
    bsz, _, height, width = pixel_values.shape
    idx = torch.randperm(bsz, device=pixel_values.device)
    x1, y1, x2, y2 = rand_bbox(height, width, lam)
    mixed = pixel_values.clone()
    mixed[:, :, y1:y2, x1:x2] = pixel_values[idx, :, y1:y2, x1:x2]
    return mixed, 1.0 - (x2 - x1) * (y2 - y1) / (height * width)

## 5. Model + Collator

In [ ]:
def make_processor(checkpoint: str) -> Mask2FormerImageProcessor:
    return Mask2FormerImageProcessor.from_pretrained(
        checkpoint,
        do_resize=False,
        do_rescale=False,
        do_normalize=False,
        reduce_labels=False,
    )


def make_collator(processor: Mask2FormerImageProcessor):
    def collate_fn(batch):
        images, mask_labels, class_labels = [], [], []
        for img, masks, labels in batch:
            images.append(img)
            mask_tensor = torch.stack([
                torch.as_tensor(m, dtype=torch.float32) for m in masks
            ])
            mask_labels.append(mask_tensor)
            class_labels.append(torch.as_tensor(labels, dtype=torch.long))
        return {
            'pixel_values': torch.stack(images),
            'mask_labels': mask_labels,
            'class_labels': class_labels,
        }
    return collate_fn


def make_model(checkpoint: str) -> Mask2FormerForUniversalSegmentation:
    return Mask2FormerForUniversalSegmentation.from_pretrained(
        checkpoint,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )

## 6. Training loop

In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, device, epoch_idx, max_batches=None):
    model.train()
    total_loss = 0.0
    n_batches = 0
    pbar = tqdm(loader, desc=f'Epoch {epoch_idx} [train]')
    for batch_idx, batch in enumerate(pbar, start=1):
        if max_batches is not None and batch_idx > max_batches:
            break
        pv = batch['pixel_values'].to(device)
        ml = [m.to(device) for m in batch['mask_labels']]
        cl = [c.to(device) for c in batch['class_labels']]
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(scaler is not None)):
            loss = model(pixel_values=pv, mask_labels=ml, class_labels=cl).loss
        if scaler:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        total_loss += loss.item()
        n_batches += 1
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    return total_loss / max(1, n_batches)


@torch.no_grad()
def validate(model, loader, device, max_batches=None):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    for batch_idx, batch in enumerate(loader, start=1):
        if max_batches is not None and batch_idx > max_batches:
            break
        pv = batch['pixel_values'].to(device)
        ml = [m.to(device) for m in batch['mask_labels']]
        cl = [c.to(device) for c in batch['class_labels']]
        total_loss += model(pixel_values=pv, mask_labels=ml, class_labels=cl).loss.item()
        n_batches += 1
    return total_loss / max(1, n_batches)


def run_training(run_name: str, cfg: dict):
    print(f'\n=== {run_name} ===')
    print(json.dumps(cfg, indent=2))
    set_seed(SEED)

    train_ds = PlantSegDataset(
        IMG_ROOT,
        train_coco,
        transform=build_transform(cfg['image_size'], cfg['aug_level'], True),
    )
    val_ds = PlantSegDataset(
        IMG_ROOT,
        val_coco,
        transform=build_transform(cfg['image_size'], 'light', False),
    )
    print(f'Dataset sizes — train: {len(train_ds)} | val: {len(val_ds)}')

    processor = make_processor(cfg['backbone'])
    collate = make_collator(processor)
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg['batch_size'],
        shuffle=True,
        num_workers=2,
        collate_fn=collate,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg['batch_size'],
        shuffle=False,
        num_workers=2,
        collate_fn=collate,
        pin_memory=torch.cuda.is_available(),
    )

    model = make_model(cfg['backbone']).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg['lr'],
        weight_decay=cfg['weight_decay'],
    )
    scaler = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None

    history, best_val, best_path = {'train_loss': [], 'val_loss': []}, float('inf'), None
    for epoch in range(1, cfg['epochs'] + 1):
        tl = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scaler,
            DEVICE,
            epoch,
            max_batches=cfg.get('max_train_batches'),
        )
        vl = validate(
            model,
            val_loader,
            DEVICE,
            max_batches=cfg.get('max_val_batches'),
        )
        history['train_loss'].append(tl)
        history['val_loss'].append(vl)
        print(f'Epoch {epoch:3d} | train {tl:.4f} | val {vl:.4f}')
        if vl < best_val:
            best_val = vl
            best_path = OUTPUT_DIR / f'{run_name}_best.pt'
            torch.save({'model_state': model.state_dict(), 'config': cfg, 'epoch': epoch}, best_path)
            print(f'  Saved: {best_path}')

    plt.figure(figsize=(8, 4))
    plt.plot(history['train_loss'], label='train')
    plt.plot(history['val_loss'], label='val')
    plt.title(f'Loss - {run_name}')
    plt.xlabel('Epoch')
    plt.legend()
    plt.savefig(OUTPUT_DIR / f'{run_name}_loss.png', dpi=100)
    plt.show()
    return model, processor, history, best_path

## 7. Evaluation — mAP, mIoU, Dice, inference time

In [ ]:
@torch.no_grad()
def evaluate_full(model, processor, coco_data, img_root, image_size: int, device, max_samples=None):
    """Evaluate on an in-memory COCO split dict."""
    model.eval()
    test_ds = PlantSegDataset(
        img_root,
        coco_data,
        transform=build_transform(image_size, 'light', False),
    )
    metric = MeanAveragePrecision(iou_type='segm')
    times_ms, iou_scores, dice_scores = [], [], []
    n_samples = len(test_ds) if max_samples is None else min(max_samples, len(test_ds))

    for idx in tqdm(range(n_samples), desc='evaluate'):
        image, gt_masks, gt_labels = test_ds[idx]
        pv = image.unsqueeze(0).to(device)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        outputs = model(pixel_values=pv)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        times_ms.append((time.perf_counter() - t0) * 1000.0)

        pred_result = processor.post_process_instance_segmentation(
            outputs,
            target_sizes=[(image_size, image_size)],
            threshold=0.5,
        )[0]

        segs = pred_result['segments_info']
        pred_masks = pred_result['segmentation']
        if len(segs) > 0:
            p_masks = torch.stack([(pred_masks == s['id']).detach().cpu().bool() for s in segs])
            p_scores = torch.tensor([s.get('score', 1.0) for s in segs], dtype=torch.float32)
            p_labels = torch.tensor([s['label_id'] for s in segs], dtype=torch.long)
        else:
            p_masks = torch.zeros(1, image_size, image_size, dtype=torch.bool)
            p_scores = torch.tensor([0.0], dtype=torch.float32)
            p_labels = torch.tensor([0], dtype=torch.long)

        g_masks = torch.stack([torch.as_tensor(m, dtype=torch.bool) for m in gt_masks])
        g_labels = torch.as_tensor(gt_labels, dtype=torch.long)

        metric.update(
            [{'masks': p_masks, 'scores': p_scores, 'labels': p_labels}],
            [{'masks': g_masks, 'labels': g_labels}],
        )
        for pm, gm in zip(p_masks[:len(gt_masks)], g_masks):
            inter = (pm & gm).sum().float()
            union = (pm | gm).sum().float()
            iou_scores.append((inter / (union + 1e-6)).item())
            dice_scores.append((2 * inter / (pm.sum() + gm.sum() + 1e-6)).item())

    res = metric.compute()
    return {
        'mAP@50': float(res.get('map_50', 0.0)),
        'mAP@50:95': float(res.get('map', 0.0)),
        'mIoU': float(np.mean(iou_scores)) if iou_scores else 0.0,
        'Dice': float(np.mean(dice_scores)) if dice_scores else 0.0,
        'inference_ms': float(np.mean(times_ms)) if times_ms else 0.0,
        'eval_samples': int(n_samples),
    }

## 8. Chạy 2 config + collect results

In [ ]:
results_summary = {}
for run_name, cfg in RUN_CONFIGS.items():
    model, processor, history, ckpt = run_training(run_name, cfg)
    metrics = evaluate_full(
        model,
        processor,
        test_coco,
        IMG_ROOT,
        cfg['image_size'],
        DEVICE,
        max_samples=cfg.get('max_eval_samples'),
    )
    results_summary[run_name] = {
        'config': cfg,
        'metrics': metrics,
        'checkpoint': str(ckpt),
        'history': history,
    }
    print(f'\n--- {run_name} metrics ---')
    print(json.dumps(metrics, indent=2))

with open(OUTPUT_DIR / 'results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

## 9. Visualization — ≥ 10 ảnh test (pred vs GT)

In [ ]:
best_run = max(results_summary, key=lambda k: results_summary[k]['metrics'].get('mAP@50:95', 0))
best_cfg = results_summary[best_run]['config']
best_model = make_model(best_cfg['backbone']).to(DEVICE)
state = torch.load(results_summary[best_run]['checkpoint'], map_location='cpu')['model_state']
best_model.load_state_dict(state)
best_processor = make_processor(best_cfg['backbone'])
best_model.eval()

viz_ds = PlantSegDataset(
    IMG_ROOT,
    test_coco,
    transform=build_transform(best_cfg['image_size'], 'light', False),
)
N_VIZ = 12
indices = np.random.choice(len(viz_ds), size=min(N_VIZ, len(viz_ds)), replace=False)
cmap = plt.get_cmap('Set1', NUM_CLASSES)

fig, axes = plt.subplots(len(indices), 3, figsize=(13, 4 * len(indices)))
if len(indices) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, i in enumerate(indices):
    image_t, gt_masks, gt_labels = viz_ds[i]
    mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std_t = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img_disp = (image_t * std_t + mean_t).permute(1, 2, 0).clamp(0, 1).numpy()

    with torch.no_grad():
        outputs = best_model(pixel_values=image_t.unsqueeze(0).to(DEVICE))
    pred = best_processor.post_process_instance_segmentation(
        outputs,
        target_sizes=[(best_cfg['image_size'], best_cfg['image_size'])],
        threshold=0.5,
    )[0]

    gt_overlay = img_disp.copy()
    for m, lbl in zip(gt_masks, gt_labels):
        color = np.array(cmap(int(lbl))[:3])
        gt_overlay[np.array(m) > 0] = gt_overlay[np.array(m) > 0] * 0.5 + color * 0.5

    pred_overlay = img_disp.copy()
    seg_map = pred['segmentation']
    if seg_map is not None and hasattr(seg_map, 'cpu'):
        seg_np = seg_map.cpu().numpy()
        for seg_info in pred['segments_info']:
            m = seg_np == seg_info['id']
            color = np.array(cmap(seg_info['label_id'])[:3])
            pred_overlay[m] = pred_overlay[m] * 0.5 + color * 0.5

    axes[row, 0].imshow(img_disp)
    axes[row, 0].set_title(f'Image #{i}')
    axes[row, 0].axis('off')
    axes[row, 1].imshow(gt_overlay)
    axes[row, 1].set_title('GT masks')
    axes[row, 1].axis('off')
    axes[row, 2].imshow(pred_overlay)
    axes[row, 2].set_title('Prediction')
    axes[row, 2].axis('off')

plt.suptitle(f'Coffee - {best_run} predictions', fontsize=14, y=1.002)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'viz_pred_vs_gt.png', dpi=100, bbox_inches='tight')
plt.show()

## 10. Error analysis

> **Điền sau khi chạy xong** — phân tích dựa trên kết quả thực.

Câu hỏi cần trả lời:
1. AlgalLeafSpot vs Rust — class nào khó phân biệt hơn qua mask? Vì sao?
2. PowderyMildew (mảng trắng phủ) — model có xu hướng over/under-segment?
3. Augmentation `strong` có giúp hay làm tệ trên coffee dataset?
4. Backbone Swin-T vs Swin-S — inference time khác nhau bao nhiêu?
5. So sánh kết quả coffee vs rice: domain nào khó hơn cho Mask2Former?

In [ ]:
# Per-class metrics (điền kết quả thực sau khi chạy)
# per_class_ap = metric.compute()['map_per_class']
# for name, ap in zip(CLASS_NAMES, per_class_ap):
#     print(f'{name}: AP={ap:.4f}')

## 11. Push best checkpoint lên HuggingFace Hub

Yêu cầu: set Kaggle Secret `HUGGINGFACE_TOKEN` trước khi chạy cell này.

In [ ]:
HF_REPO = os.environ.get('HF_REPO', 'tunah/mask2former-coffee-seg')

try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login

    hf_token = UserSecretsClient().get_secret('HUGGINGFACE_TOKEN')
    if not hf_token:
        raise ValueError('Kaggle Secret HUGGINGFACE_TOKEN is empty')

    login(token=hf_token)
    print(f'Best run: {best_run} | mAP@50:95 = {results_summary[best_run]["metrics"]["mAP@50:95"]:.4f}')
    best_model.push_to_hub(HF_REPO)
    best_processor.push_to_hub(HF_REPO)
    print(f'Pushed to: https://huggingface.co/{HF_REPO}')
except Exception as exc:
    print('Skip HuggingFace push:', repr(exc))
    print('Set Kaggle Secret HUGGINGFACE_TOKEN and optionally env HF_REPO, then rerun this cell.')

## 12. Bảng so sánh trước/sau tuning

In [ ]:
rows = []
for run_name, r in results_summary.items():
    row = {'run': run_name, **r['metrics']}
    row.update({k: r['config'][k] for k in ['backbone', 'image_size', 'batch_size', 'lr', 'aug_level']})
    rows.append(row)
df_compare = pd.DataFrame(rows)
df_compare.to_csv(OUTPUT_DIR / 'comparison_coffee.csv', index=False)

print('=== Bảng so sánh 2 config — Coffee ===')
df_compare